# ZİHİN — Colab çağırıcısı

**Bu defter iş yapmıyor, çağırıyor.** Kod depoda ve sürümlü. Kurulum,
jeton adımları ve çalıştırma sırası: yanındaki `README.md`.

## Önce bunlar

1. **GitHub jetonu.** Colab'ın GitHub yetkilendirmesi yetmiyor — o defteri
   *açmaya* yarıyor. Klonlama ve sonuç itme ayrı bir jeton istiyor:
   fine-grained, yalnız `zihin` deposu, **Contents: Read and write**.
   Adımlar `README.md` içinde.
2. **Colab Secrets** (soldaki anahtar simgesi) → `GITHUB_TOKEN` adıyla ekleyin
   ve **Notebook access**'i açın.
3. **Çalışma zamanı → A100 GPU.** Şartlar tablosu (`runtime`, §19.6) A100
   diyor: ilan edilen modelin ölçülen dosyası 19,851 GB ve T4'ün 15360
   MiB'ine sığmıyor. T4'te koşu, dokuz gigabaytlık değil yirmi gigabaytlık
   bir çekimden *sonra* yerleşiklik denetiminde düşer.

## Çalıştırma sırası — *tümünü çalıştır* değil

**1 → 3'ü çalıştırın, ortam ölçümüne bakın, sonra 4 ve 5.** Dilim başlamadan
ölçüm çıktısına bakılması isteniyor. Dilimin boyutu dördüncü hücredeki
`--tasks`; ilk dilimi Pro+ tahsisine göre siz seçiyorsunuz (§19.6). Dördüncü
hücre kendini de koruyor: ölçüm yoksa ya da model tamamen VRAM'de değilse
başlamıyor.


## 1. Drive

sqlite ve model önbelleği burada. Olmazsa dokuz gigabayt her oturum yeniden
çekilir ve sonuç oturumla kaybolur.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Jeton, klon, önyükleme

Depo özel: klonlama jeton istiyor. Gizli anahtar yoksa bu hücre **ne
yapılacağını söyleyerek** duruyor — git'in anlaşılmaz hatasıyla değil.


In [ ]:
import json
import os
import pathlib
import subprocess
import sys
import urllib.error
import urllib.request

SLUG = os.environ.get("ZIHIN_REPO_SLUG", "yusuferengerek/zihin")
# The branch to clone. One place, and `tests/test_colab_ref.py`
# pins that the README names the same one: the notebook used to
# default to a feature branch and kept doing so after that branch
# merged, so a clean Colab checkout came up missing files that were
# in `main` — and the failure named itself "the agent forgot" rather
# than "this copy is stale".
REF = os.environ.get("ZIHIN_REF", "main")
WORKDIR = os.environ.get("ZIHIN_WORKDIR", "/content/zihin")
API = "https://api.github.com"

HELP = """
GITHUB_TOKEN yok.

Depo özel; klonlamak ve sonuçları itmek için jeton gerekiyor.
Colab'ın GitHub yetkilendirmesi bunun yerine geçmiyor — o, defteri açmaya yarıyor.

1) github.com/settings/personal-access-tokens/new  ->  fine-grained
2) Only select repositories  ->  zihin
3) Repository permissions -> Contents: Read and write
4) Colab'da soldaki anahtar simgesi -> Add new secret
   Name: GITHUB_TOKEN,  Notebook access açık

Ayrıntı: experiments/colab/README.md
"""


def secret():
    """Reads the token: Colab's store first, the environment second.

    The environment branch is not a test hatch. It is what makes every line
    below runnable outside Colab, and this step is not Colab-specific — only
    the place the secret lives is. Without it the step ships untested, which is
    the thing K-122 was written about.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return os.environ.get("GITHUB_TOKEN", "").strip()
    try:
        return (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception as error:
        print("Colab Secrets okunamadı:", error)
        return ""


TOKEN = secret()
if not TOKEN:
    sys.exit(HELP)
os.environ["GITHUB_TOKEN"] = TOKEN


def ask(path):
    """One GitHub API call. Returns (status, body) and never raises."""
    request = urllib.request.Request(
        API + path,
        headers={
            "Authorization": "Bearer " + TOKEN,
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=30) as answer:
            return answer.status, json.load(answer)
    except urllib.error.HTTPError as error:
        return error.code, None
    except Exception as error:
        return -1, str(error)


# Cheapest and most local check first: this one needs no token and no network.
target = pathlib.Path(WORKDIR)
if target.exists() and not (target / ".git").is_dir():
    inside = sorted(entry.name for entry in target.iterdir())[:10]
    sys.exit(f"""GÖZLENDİ: {WORKDIR} var ama checkout değil. İçinde: {inside}
Klon boş olmayan bir dizine yazamıyor. Silip hücreyi yeniden çalıştırın:
    !rm -rf {WORKDIR}""")

# The preflight. `git clone` reports at least four different failures with the
# same exit code 128 — destination not empty, invalid token, no access, no such
# branch — and the number separates none of them; all four were reproduced in a
# stripped local environment. These calls separate them before git runs, and
# each one states what was observed rather than a cause someone guessed (K-123).
status, _ = ask("/user")
if status == 401:
    sys.exit(f"""GÖZLENDİ: GitHub /user -> 401.
Jeton geçersiz, süresi dolmuş ya da iptal edilmiş. Yeni bir jeton oluşturup
Colab Secrets'taki GITHUB_TOKEN değerini güncelleyin.
{HELP}""")
if status != 200:
    print(f"not: /user -> {status} (beklenen 200). Yorumlanmıyor; kontrol sürüyor.")

status, repository = ask("/repos/" + SLUG)
if status == 404:
    sys.exit(f"""GÖZLENDİ: GitHub /repos/{SLUG} -> 404.
Jeton geçerli ama bu depoyu göremiyor. GitHub, göremediği özel bir depoya 403
değil 404 döndürüyor: bu "yok" değil "yetkin yok" demek.
Jetonda Repository access -> Only select repositories -> zihin seçili mi, ve
Contents: Read and write verilmiş mi, bakın.""")
if status != 200:
    sys.exit(f"GÖZLENDİ: GitHub /repos/{SLUG} -> {status}. Beklenen 200; klon denenmedi.")
print("depo görünüyor:", repository["full_name"])

status, _ = ask("/repos/" + SLUG + "/branches/" + REF)
if status != 200:
    sys.exit(f"GÖZLENDİ: {REF} dalı -> {status}. Dal adı yanlış ya da dal itilmemiş.")
print("dal görünüyor:", REF)

# The token goes through an askpass helper: never in the URL, never in `argv`.
# Measured the hard way. The first form carried it inside the clone URL, the
# clone failed, and `CalledProcessError` prints `process.args` — so the secret
# left the runtime inside a traceback and had to be revoked. git redacts the URL
# in its own messages; Python does not (K-124).
ASKPASS = pathlib.Path("/tmp/zihin-askpass.sh")
ASKPASS.write_text(
    """#!/bin/sh
case "$1" in
  Username*) echo "x-access-token" ;;
  *) echo "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8",
)
ASKPASS.chmod(0o700)

# The helper goes into the process environment too, not only into the
# dict this cell hands to git. Every later cell and every subprocess -
# the preflight's `git fetch` among them - inherits `os.environ`, and it
# was inheriting the token with no way for git to use it: `GITHUB_TOKEN`
# was exported, `GIT_ASKPASS` was not. The fetch had a credential in the
# environment and no helper to hand it over, so it failed and the
# staleness check answered "unknown" in the one environment where
# staleness is most likely (K-152).
os.environ["GIT_ASKPASS"] = str(ASKPASS)
os.environ["GIT_TERMINAL_PROMPT"] = "0"
# The working directory goes into the environment for the same reason the
# askpass helper does: `colab_bootstrap.sh` and `colab_publish_results.sh`
# both read `ZIHIN_WORKDIR` and both defaulted to `/content/zihin`, so an
# override honoured here was ignored three cells later. One value, one
# place, and the later cells address it as `$WORKDIR`.
os.environ["ZIHIN_WORKDIR"] = WORKDIR
ENVIRONMENT = dict(
    os.environ,
    GITHUB_TOKEN=TOKEN,
    GIT_ASKPASS=str(ASKPASS),
    GIT_TERMINAL_PROMPT="0",
)
URL = "https://github.com/" + SLUG + ".git"


def git(*arguments):
    """Runs git and, when it fails, shows git's own sentence.

    The previous form used `check=True`, so a failure surfaced as a number with
    the message stripped off. Exit 128 has at least four causes here and the
    number distinguishes none of them (K-125).
    """
    done = subprocess.run(arguments, env=ENVIRONMENT, capture_output=True, text=True)
    if done.returncode != 0:
        joined = " ".join(arguments)
        said = (done.stderr or done.stdout).strip() or "(git hiçbir şey yazmadı)"
        sys.exit(f"""GÖZLENDİ: {joined}
  çıkış {done.returncode}
  {said.replace(TOKEN, "<jeton>")}""")
    return done.stdout.strip()


if (target / ".git").is_dir():
    git("git", "-C", WORKDIR, "remote", "set-url", "origin", URL)
    git("git", "-C", WORKDIR, "fetch", "-q", "origin", REF)
    git("git", "-C", WORKDIR, "checkout", "-q", REF)
    # A hard reset rather than a pull: the checkout is disposable, the record
    # lives on Drive and on the results branch, and a pull can stop on a
    # divergence nobody is here to resolve.
    git("git", "-C", WORKDIR, "reset", "-q", "--hard", "origin/" + REF)
else:
    git("git", "clone", "-q", "--branch", REF, URL, WORKDIR)

print("klon tamam:", git("git", "-C", WORKDIR, "rev-parse", "--short", "HEAD"), REF)

In [ ]:
# uv run re-syncs before every command. If the AppWorld mirror is in use
# that re-sync returns to the upstream source and fails; set here so every
# later cell inherits it from the kernel.
%env UV_NO_SYNC=1
# The manifest reads the server's settings from this file. Every cell is its
# own shell, so the bootstrap's export does not reach the run.
%env ZIHIN_SERVER_LOG=/tmp/zihin-ollama.log
# Set once, here, because every cell is its own shell. The model is NOT set:
# it is resolved from the conditions the protocol declares for this run's
# prefix, so the name lives in exactly one place (K-138).
%env MODEL_SERVER_URL=http://127.0.0.1:11434
!bash $WORKDIR/scripts/colab_bootstrap.sh

## 3. Ortam ölçümü — kontrol noktası

GPU ve VRAM, modelin tamamen VRAM'de olup olmadığı, yükleme süresi.
**Buraya bakın, sonra devam edin.** Tam yerleşik değilse çıkış kodu 1.


In [ ]:
%cd $WORKDIR
!uv run python scripts/setup/measure_colab_environment.py --repeats 3

## 4. Deney, faz 1 — şartlar §19.6'da mutlak değerle

Ölçüm koşmamışsa ya da model tam yerleşik değilse bu hücre başlamıyor.
`--tasks` dilimin boyutu: bildirilen sıranın sıradaki o kadar görevi.
Oturum kesilirse aynı hücreyi yeniden çalıştırın: yarım dilim önce
tamamlanıyor, yazılmış epizotlar atlanıyor (§19.6, `tests/test_run_experiment.py`).
Ölçümü tanımlayan bir yol ilk epizottan beri değiştiyse hücre duruyor;
devam etmek tarihli bir ek istiyor.


In [ ]:
%cd $WORKDIR
# One line, chained. Two `!` lines would not have been a guard: a failing
# shell escape does not stop the next one and the cell still reports success -
# measured, and the runner started after the guard had refused.
#
# The experiment's phase 1, declared in section 19.6. The prefix names its own
# conditions section, and the driver resolves the model and every condition
# from there (K-138, K-185). `--tasks` is the slice's size: the next that many
# tasks of the declared order. It is not a condition.
!uv run python scripts/setup/require_colab_environment.py && uv run python scripts/run_experiment.py dny --tasks 10

## 5. Sonuçları depoya it

`main`'e değil `colab-sonuclari` dalına, zorla itme yok, yalnız `results/`
altı. İtmenin indiği uzak SHA ile doğrulanıyor (§15.7, K-116 - koşudan bağımsız kural, §19'un kendi bölümü yok).


In [ ]:
# The database is already on Drive: the bootstrap links it there. Copying a
# link onto its own target was a no-op that read like a safeguard.
# No prefix argument: the script resolves it from the run's declaration,
# so the prefix is written once, in the run command above.
!bash $WORKDIR/scripts/colab_publish_results.sh